In [10]:
import sqlite3
from werkzeug.security import generate_password_hash

def inicializar_sistema():
    conn = sqlite3.connect('biblioteca.db')
    cursor = conn.cursor()

    # Tabela de Usuários (Admin e Estudantes)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Usuario (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            senha TEXT NOT NULL,
            tipo TEXT DEFAULT 'estudante'
        )
    ''')

    # Tabela de Livros (com os campos oficiais da EMEF)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Livro (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            autor TEXT NOT NULL,
            editora TEXT,
            ano INTEGER,
            quantidade INTEGER DEFAULT 1,
            localizacao TEXT, -- Ex: Corredor A, Estante 1
            status TEXT DEFAULT 'disponivel'
        )
    ''')

    # Tabela de Empréstimos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Emprestimo (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            livro_id INTEGER,
            usuario_id INTEGER,
            data_emprestimo TEXT,
            data_devolucao TEXT,
            status TEXT DEFAULT 'ativo',
            FOREIGN KEY(livro_id) REFERENCES Livro(id),
            FOREIGN KEY(usuario_id) REFERENCES Usuario(id)
        )
    ''')

    # Criar Administrador padrão se não existir
    cursor.execute("SELECT * FROM Usuario WHERE email = 'admin@escola.com'")
    if not cursor.fetchone():
        senha_hash = generate_password_hash("admin123")
        cursor.execute("INSERT INTO Usuario (nome, email, senha, tipo) VALUES (?, ?, ?, ?)",
                       ("Bibliotecário(a)", "admin@escola.com", senha_hash, "admin"))

    conn.commit()
    conn.close()
    print("✅ Banco de dados inicializado com sucesso!")

inicializar_sistema()

✅ Banco de dados inicializado com sucesso!


In [11]:
from datetime import datetime, timedelta

def cadastrar_livro_teste(titulo, autor, editora, ano, quantidade, localizacao):
    conn = sqlite3.connect('biblioteca.db')
    cursor = conn.cursor()
    try:
        cursor.execute('''
            INSERT INTO Livro (titulo, autor, editora, ano, quantidade, localizacao)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (titulo, autor, editora, ano, quantidade, localizacao))
        conn.commit()
        print(f"✅ Livro '{titulo}' cadastrado com sucesso na localização {localizacao}!")
    except Exception as e:
        print(f"❌ Erro ao cadastrar: {e}")
    finally:
        conn.close()

def listar_livros_db():
    conn = sqlite3.connect('biblioteca.db')
    cursor = conn.cursor()
    cursor.execute("SELECT id, titulo, autor, localizacao, status FROM Livro")
    livros = cursor.fetchall()
    conn.close()

    print("\n📚 ACERVO ATUAL DA BIBLIOTECA:")
    for l in livros:
        print(f"ID: {l[0]} | Título: {l[1]} | Autor: {l[2]} | Local: {l[3]} | Status: {l[4]}")

def realizar_emprestimo(livro_id, usuario_id):
    conn = sqlite3.connect('biblioteca.db')
    cursor = conn.cursor()

    # Verifica se o livro está disponível
    cursor.execute("SELECT status FROM Livro WHERE id = ?", (livro_id,))
    livro = cursor.fetchone()

    if livro and livro[0] == 'disponivel':
        data_emp = datetime.now().strftime('%Y-%m-%d')
        data_dev = (datetime.now() + timedelta(days=14)).strftime('%Y-%m-%d') # Prazo de 14 dias

        cursor.execute('''
            INSERT INTO Emprestimo (livro_id, usuario_id, data_emprestimo, data_devolucao, status)
            VALUES (?, ?, ?, ?, 'ativo')
        ''', (livro_id, usuario_id, data_emp, data_dev))

        cursor.execute("UPDATE Livro SET status = 'emprestado' WHERE id = ?", (livro_id,))
        conn.commit()
        print(f"✅ Empréstimo realizado com sucesso! Devolução prevista para: {data_dev}")
    else:
        print("❌ Livro não disponível para empréstimo.")
    conn.close()

print("✅ Funções de lógica carregadas!")

✅ Funções de lógica carregadas!


In [12]:
# 1. Cadastrar um livro de teste espelhando o padrão da escola
cadastrar_livro_teste(
    titulo="O Alienista",
    autor="Machado de Assis",
    editora="Ática",
    ano=2005,
    quantidade=2,
    localizacao="Corredor A - Estante 1"
)

# 2. Listar os livros cadastrados
listar_livros_db()

# 3. Simular um empréstimo do livro ID 1 para o usuário ID 1 (o admin ou um aluno cadastrado)
realizar_emprestimo(livro_id=1, usuario_id=1)

# 4. Listar novamente para ver o status mudando para 'emprestado'
listar_livros_db()

✅ Livro 'O Alienista' cadastrado com sucesso na localização Corredor A - Estante 1!

📚 ACERVO ATUAL DA BIBLIOTECA:
ID: 1 | Título: O Alienista | Autor: Machado de Assis | Local: Corredor A - Estante 1 | Status: disponivel
✅ Empréstimo realizado com sucesso! Devolução prevista para: 2026-11-14

📚 ACERVO ATUAL DA BIBLIOTECA:
ID: 1 | Título: O Alienista | Autor: Machado de Assis | Local: Corredor A - Estante 1 | Status: emprestado
